# 🌍 Explorador API ODS — INEGI
Scraper exploratorio para entender qué datos trae la API pública de Indicadores ODS del INEGI.

**Base URL:** `https://ods.org.mx/api/`

In [1]:
import requests
import pandas as pd
import json
from IPython.display import display

BASE_URL = "https://ods.org.mx/api"
HEADERS = {"Content-Type": "application/json"}

def post(endpoint, params):
    r = requests.post(f"{BASE_URL}/{endpoint}", json=params, headers=HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

## 1. Catálogo completo — estructura ODS
Traemos toda la jerarquía: Objetivos → Metas → Indicadores

In [2]:
catalogo_raw = post("Tematica/Todos", {"PIdioma": "es"})
print(f"Objetivos encontrados: {len(catalogo_raw)}")
print(f"\nEjemplo — primer objetivo:")
print(json.dumps(catalogo_raw[0], indent=2, ensure_ascii=False)[:600], "...")

Objetivos encontrados: 17

Ejemplo — primer objetivo:
{
  "Clave_arb": "ODS0010",
  "Codigo_des": "1.",
  "Descrip_des": "Poner fin a la pobreza en todas sus formas y en todo el mundo",
  "Abrevia_des": "Fin de la pobreza",
  "NumIndicadores": 31,
  "Ind_disponibles": true,
  "Meta": [
    {
      "Clave_arb": "ODS00100010",
      "Codigo_des": "Meta 1.1",
      "Descrip_des": "De aquí a 2030, erradicar para todas las personas y en todo el mundo la pobreza extrema (actualmente se considera que sufren pobreza extrema las personas que viven con menos de 1,25 dólares de los Estados Unidos al día)",
      "Abrevia_des": "",
      "NumIndicadores": 3, ...


## 2. Aplanar el catálogo a un DataFrame
Extraemos todos los indicadores con su metadata relevante.

In [3]:
rows = []

for objetivo in catalogo_raw:
    for meta in objetivo.get("Meta", []):          # era "Indicador", es "Meta"
        for ind in meta.get("Indicador", []):
            tipos = ", ".join([t.get("DescripS_tatr", "") for t in ind.get("TipoInd", [])])
            rows.append({
                "objetivo_clave": objetivo.get("Clave_arb"),
                "objetivo_desc": objetivo.get("Abrevia_des"),  # nombre corto, ej "Fin de la pobreza"
                "meta_clave": meta.get("Clave_arb"),
                "meta_codigo": meta.get("Codigo_des"),
                "meta_desc": meta.get("Descrip_des"),
                "indicador_clave": ind.get("Clave_arb"),
                "indicador_codigo": ind.get("Codigo_des"),
                "indicador_desc": ind.get("Descrip_des"),
                "clave_serie": ind.get("Clave_ser"),
                "num_registros": ind.get("NumRegistros", 0),
                "tiene_valores": ind.get("TieneValores", False),
                "tipo": tipos,
                "desglose_geo": ind.get("DesGeo", {}).get("Descrip_dg", ""),
            })

df = pd.DataFrame(rows)
print(f"Total indicadores: {len(df)}")
print(f"Con valores disponibles: {df['tiene_valores'].sum()}")
display(df.head(10))

Total indicadores: 336
Con valores disponibles: 336


,objetivo_clave,objetivo_desc,meta_clave,meta_codigo,meta_desc,indicador_clave,indicador_codigo,indicador_desc,clave_serie,num_registros,tiene_valores,tipo,desglose_geo
0,ODS0010,Fin de la pobreza,ODS00100010,Meta 1.1,"De aquí a 2030, erradicar para todas las perso...",ODS001000100010,1.1.1,Proporción de la población que vive por debajo...,355,48,True,Global,Sin desglose geográfico
1,ODS0010,Fin de la pobreza,ODS00100010,Meta 1.1,"De aquí a 2030, erradicar para todas las perso...",ODS001000100020,1.1.1.a,Proporción de la población que vive por debajo...,1080,99,True,Global,Desglose por entidad federativa
2,ODS0010,Fin de la pobreza,ODS00100010,Meta 1.1,"De aquí a 2030, erradicar para todas las perso...",ODS001000100030,1r.1.1,Porcentaje de la población que vive por debajo...,1873,48,True,Regional,Sin desglose geográfico
3,ODS0010,Fin de la pobreza,ODS00100020,Meta 1.2,"De aquí a 2030, reducir al menos a la mitad la...",ODS001000200002,1.2.1(1),Proporción de la población que vive por debajo...,640,36,True,Global,Sin desglose geográfico
4,ODS0010,Fin de la pobreza,ODS00100020,Meta 1.2,"De aquí a 2030, reducir al menos a la mitad la...",ODS001000200004,1.2.1(2),Proporción de la población que vive por debajo...,1610,36,True,Global,Sin desglose geográfico
5,ODS0010,Fin de la pobreza,ODS00100020,Meta 1.2,"De aquí a 2030, reducir al menos a la mitad la...",ODS001000200005,1.2.1.a,Proporción de la población que vive por debajo...,2194,132,True,Global,Desglose por entidad federativa
6,ODS0010,Fin de la pobreza,ODS00100020,Meta 1.2,"De aquí a 2030, reducir al menos a la mitad la...",ODS001000200007,1.2.1.b,Proporción de la población que vive por debajo...,2174,7509,True,Global,Desglose por entidad federativa y municipio
7,ODS0010,Fin de la pobreza,ODS00100020,Meta 1.2,"De aquí a 2030, reducir al menos a la mitad la...",ODS001000200010,1.2.2(1),"Proporción de hombres, mujeres y niños de toda...",429,36,True,Global,Sin desglose geográfico
8,ODS0010,Fin de la pobreza,ODS00100020,Meta 1.2,"De aquí a 2030, reducir al menos a la mitad la...",ODS001000200015,1.2.2(2),"Proporción de hombres, mujeres y niños de toda...",1566,36,True,Global,Sin desglose geográfico
9,ODS0010,Fin de la pobreza,ODS00100020,Meta 1.2,"De aquí a 2030, reducir al menos a la mitad la...",ODS001000200020,1.2.2.a,"Proporción de hombres, mujeres y niños de toda...",1539,132,True,Global,Desglose por entidad federativa


## 3. Vista general — indicadores por ODS

In [4]:
resumen = df.groupby("objetivo_desc").agg(
    total_indicadores=("indicador_clave", "count"),
    con_valores=("tiene_valores", "sum"),
    total_registros=("num_registros", "sum")
).reset_index()

resumen["cobertura_%"] = (resumen["con_valores"] / resumen["total_indicadores"] * 100).round(1)
display(resumen.sort_values("total_registros", ascending=False))

,objetivo_desc,total_indicadores,con_valores,total_registros,cobertura_%
13,Salud y bienestar,42,42,663678,100.0
4,Educación de calidad,33,33,284930,100.0
6,Fin de la pobreza,31,31,171908,100.0
7,Hambre cero,17,17,92156,100.0
14,Trabajo decente y crecimiento económico,31,31,78118,100.0
3,Ciudades y comunidades sostenibles,16,16,43188,100.0
10,"Paz, justicia e instituciones sólidas",36,36,35221,100.0
8,Igualdad de género,29,29,10900,100.0
9,"Industria, innovación e infraestructura",16,16,6773,100.0
12,Reducción de las desigualdades,11,11,2487,100.0


## 4. Distribución por tipo de indicador

In [5]:
print(df["tipo"].value_counts().to_string())
print("\nDesglose geográfico disponible:")
print(df["desglose_geo"].value_counts().to_string())

tipo
Global      216
Nacional     86
Regional     34

Desglose geográfico disponible:
desglose_geo
Sin desglose geográfico                                                      180
Desglose por entidad federativa                                              132
Desglose por entidad federativa y municipio                                   15
Desglose por entidad federativa y principales ciudades                         5
Desglose por entidad federativa y áreas metropolitanas                         1
Proporción plazas públicas                                                     1
Plazas en el Poder judicial                                                    1
Desglose por entidad federativa y áreas metropolitanas autorrepresentadas      1


## 5. Preview de valores reales
Tomamos una muestra de indicadores con valores y pedimos sus datos históricos.

In [10]:
# Muestra: 20 indicadores con valores, priorizando los que tienen más registros
muestra = (
    df[df["tiene_valores"] == True]
    .sort_values("num_registros", ascending=False)
    .dropna(subset=["clave_serie"])
    .head(20)
)

print("Indicadores seleccionados para preview:")
display(muestra[["objetivo_desc", "indicador_codigo", "indicador_desc", "clave_serie", "num_registros"]])

Indicadores seleccionados para preview:


,objetivo_desc,indicador_codigo,indicador_desc,clave_serie,num_registros
63,Salud y bienestar,3.4.1,Tasa de mortalidad atribuida a las enfermedade...,2307,178200
64,Salud y bienestar,3.4.2.a,Tasa de mortalidad por suicidio,2277,81840
55,Salud y bienestar,3.3.2c,"Incidencia de la tuberculosis por cada 100,000...",2165,78408
66,Salud y bienestar,3.6.1,Tasa de mortalidad por lesiones debidas a acci...,543,75900
30,Fin de la pobreza,1n.3.3.b,Porcentaje de la población que presenta carenc...,1728,75090
44,Hambre cero,2n.2.1.b,Porcentaje de población con inseguridad alimen...,1719,75090
81,Salud y bienestar,3n.1.1.b,Porcentaje de la población que presenta carenc...,2152,75090
21,Fin de la pobreza,1n.1.1.b,Porcentaje de la población en situación de pob...,1725,75090
74,Salud y bienestar,3.9.2a,Tasa de mortalidad atribuida al agua insalubre...,1569,68640
76,Salud y bienestar,3.9.3a,Tasa de mortalidad atribuida a intoxicaciones ...,1548,68640


In [7]:
def fetch_valores(clave_ind, clave_serie, ano_ini=2000, ano_fin=2024):
    """Pide valores históricos de una serie específica."""
    payload = {
        "PCveInd": str(int(clave_ind)) if pd.notna(clave_ind) else "",
        "PClaveSer": str(int(clave_serie)),
        "PAñoInicial": str(ano_ini),
        "PAñoFinal": str(ano_fin),
        "POrden": "ASC",
        "PIdioma": "es"
    }
    url = f"{BASE_URL}/Valores/PorClaveSerie"
    print(f"  → POST {url}")
    print(f"  → payload: {json.dumps(payload, ensure_ascii=False)}")
    
    try:
        r = requests.post(url, json=payload, headers=HEADERS, timeout=15)
        print(f"  → status: {r.status_code}")
        print(f"  → respuesta raw: {r.text[:300]}")
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {"error": str(e)}

previews = {}
for _, row in muestra.iterrows():
    label = f"{row['indicador_codigo']} — {row['indicador_desc'][:60]}..."
    print(f"\nFetching: {label}")
    print(f"  → clave_ind que se usa: {row.get('clave_ind', 'NO EXISTE')}  |  clave_serie: {row['clave_serie']}")
    
    result = fetch_valores(
        row.get("clave_ind", row["clave_serie"]),  # fallback hasta que tengamos clave_ind
        row["clave_serie"]
    )
    previews[label] = result
    
    if isinstance(result, list) and len(result) > 0:
        print(f"  ✓ {len(result)} registros")
        print(f"  Primer registro: {json.dumps(result[0], ensure_ascii=False, indent=2)[:300]}")
    else:
        print(f"  ✗ Respuesta: {result}")


Fetching: 3.4.1 — Tasa de mortalidad atribuida a las enfermedades cardiovascul...
  → clave_ind que se usa: NO EXISTE  |  clave_serie: 2307
  → POST https://ods.org.mx/api/Valores/PorClaveSerie
  → payload: {"PCveInd": "2307", "PClaveSer": "2307", "PAñoInicial": "2000", "PAñoFinal": "2024", "POrden": "ASC", "PIdioma": "es"}
  → status: 400
  → respuesta raw: {"Message":"Usuario o contaseña NO válido"}
  ✗ Respuesta: {'error': '400 Client Error: Bad Request for url: https://ods.org.mx/api/Valores/PorClaveSerie'}

Fetching: 3.4.2.a — Tasa de mortalidad por suicidio...
  → clave_ind que se usa: NO EXISTE  |  clave_serie: 2277
  → POST https://ods.org.mx/api/Valores/PorClaveSerie
  → payload: {"PCveInd": "2277", "PClaveSer": "2277", "PAñoInicial": "2000", "PAñoFinal": "2024", "POrden": "ASC", "PIdioma": "es"}
  → status: 400
  → respuesta raw: {"Message":"Usuario o contaseña NO válido"}
  ✗ Respuesta: {'error': '400 Client Error: Bad Request for url: https://ods.org.mx/api/Valores/PorClav

## 6. Exploración manual — busca indicadores por texto
Útil para encontrar indicadores relevantes para tu caso de estudio.

In [8]:
# Cambia este texto para buscar lo que te interese
BUSQUEDA = "pobreza"

resultado = df[
    df["indicador_desc"].str.contains(BUSQUEDA, case=False, na=False) |
    df["meta_codigo"].str.contains(BUSQUEDA, case=False, na=False)
]

print(f"Indicadores que contienen '{BUSQUEDA}': {len(resultado)}")
display(resultado[["objetivo_desc", "indicador_codigo", "indicador_desc", "clave_serie", "tiene_valores", "num_registros"]])

Indicadores que contienen 'pobreza': 17


,objetivo_desc,indicador_codigo,indicador_desc,clave_serie,tiene_valores,num_registros
0,Fin de la pobreza,1.1.1,Proporción de la población que vive por debajo...,355,True,48
1,Fin de la pobreza,1.1.1.a,Proporción de la población que vive por debajo...,1080,True,99
2,Fin de la pobreza,1r.1.1,Porcentaje de la población que vive por debajo...,1873,True,48
3,Fin de la pobreza,1.2.1(1),Proporción de la población que vive por debajo...,640,True,36
4,Fin de la pobreza,1.2.1(2),Proporción de la población que vive por debajo...,1610,True,36
5,Fin de la pobreza,1.2.1.a,Proporción de la población que vive por debajo...,2194,True,132
6,Fin de la pobreza,1.2.1.b,Proporción de la población que vive por debajo...,2174,True,7509
7,Fin de la pobreza,1.2.2(1),"Proporción de hombres, mujeres y niños de toda...",429,True,36
8,Fin de la pobreza,1.2.2(2),"Proporción de hombres, mujeres y niños de toda...",1566,True,36
9,Fin de la pobreza,1.2.2.a,"Proporción de hombres, mujeres y niños de toda...",1539,True,132


## 7. Guardar catálogo completo
Para no tener que volver a pedirlo cada vez.

In [9]:
df.to_csv("catalogo_ods_inegi.csv", index=False, encoding="utf-8-sig")
print("Guardado: catalogo_ods_inegi.csv")
print(f"Shape: {df.shape}")

Guardado: catalogo_ods_inegi.csv
Shape: (336, 13)
